# Week Assignment 02 — DuckDB Feature Aggregation

This notebook aggregates the raw daily rows into page-level features using DuckDB (`REGR_SLOPE`, window filters), following the workflow from Starter Notebook 03. It also defines the forward-looking label and documents the leakage-safe, out-of-time split design.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.features import build_train_valid, TRAIN_WINDOW, VALID_WINDOW

raw = pd.read_csv('../data/sample_search_data.csv', parse_dates=['date'])
TRAIN_WINDOW, VALID_WINDOW

(WindowSpec(name='train', feat_start=31, feat_end=90, label_start=91, label_end=150),
 WindowSpec(name='valid', feat_start=61, feat_end=120, label_start=121, label_end=180))

## Split design
- **Train**: features from day 31-90 → label (click growth) from day 91-150
- **Valid**: features from day 61-120 → label (click growth) from day 121-180

The validation feature window starts *after* the training feature window ends, and both label windows are strictly in the future relative to their own feature window — so no row ever uses information from its own future, and the validation cohort is evaluated on a period the training process never saw.

In [2]:
train_df, valid_df = build_train_valid(raw)
train_df.shape, valid_df.shape

((320, 17), (320, 17))

In [3]:
train_df[['page_id','avg_position_mean','position_slope','ctr_actual',
          'ctr_gap_vs_expected','click_growth_rate','momentum_class']].head(8)

,page_id,avg_position_mean,position_slope,ctr_actual,ctr_gap_vs_expected,click_growth_rate,momentum_class
0,page_0000,5.268500,-0.007231,0.048206,-0.001794,0.017995,stable
1,page_0001,12.314833,0.029784,0.015055,-0.000545,-0.303176,declining
2,page_0002,17.622000,-0.002272,0.008574,0.000174,-0.008696,stable
3,page_0003,7.184667,0.001069,0.029819,-0.000181,-0.011111,stable
4,page_0004,10.486500,0.023606,0.017462,-0.000538,-0.267857,declining
5,page_0005,15.478167,0.043625,0.011410,-0.000590,0.020408,stable
6,page_0006,9.528833,0.015920,0.018441,0.000441,-0.200000,declining
7,page_0007,13.436000,0.031105,0.014039,-0.000361,-0.360129,declining


## Label balance

In [4]:
train_df.momentum_class.value_counts(), valid_df.momentum_class.value_counts()

(momentum_class
 stable       156
 growing       84
 declining     80
 Name: count, dtype: int64,
 momentum_class
 stable       149
 growing       93
 declining     78
 Name: count, dtype: int64)

## Sanity check against synthetic ground truth
*(offline check only — `true_regime` never enters the model)*

In [5]:
check = raw[['page_id','true_regime']].drop_duplicates().merge(train_df, on='page_id')
pd.crosstab(check.true_regime, check.momentum_class)

momentum_class,declining,growing,stable
true_regime,,,
declining,79,0,3
growing,0,82,1
recovering,1,1,54
stable,0,1,98


## Takeaways
- The derived `momentum_class` label lines up well with the synthetic ground-truth regime (growing pages mostly land in `growing`, declining pages mostly in `declining`), which validates the label definition before any model is trained on it.
- `position_slope`, `ctr_gap_vs_expected`, and `impressions_slope` look like the most information-dense features going into modeling.